# Assignment 2A — Part A: Prompt Paradigms & System Design  [5 Marks]

**Domain:** Medical & Clinical Literature (same corpus as Assignment 1A)
**SFT base:** best QLoRA adapter from Assignment 1 Part B (`outputs/qlora_adapters/adapter_B`)
**Decoding:** greedy (`do_sample=False`) as carried over from Assignment 1A, `max_new_tokens=200`;
self-consistency is the one deliberate exception at `temperature=0.7`.

| Step | Deliverable | Marks |
|---|---|---|
| A1 | 5 prompting strategies × 5 domain queries + quality comparison | 3 |
| A2 | 2-step prompt chain + function-calling JSON output + comparison | 2 |


> **Backend switch.** Every cell runs with `BACKEND = "offline"`, a deterministic
> corpus-grounded simulator, so the tables, the chain comparison, the PAL execution and the
> preference dataset can be produced and checked on CPU with no model download. For the marked
> run, open a GPU runtime, set `BACKEND = "adapter"`, and re-run; the code path is identical and
> every saved artefact records which backend produced it.


In [ ]:
# Run once in Colab:
# !pip install -q -r assignment2a/requirements_colab.txt


In [ ]:
from pathlib import Path
import sys, json

A2 = Path.cwd()
if (A2 / "assignment2a").exists():
    A2 = A2 / "assignment2a"
sys.path.insert(0, str(A2 / "src"))

import a2_paths
from a2_paths import DEFAULT_SFT_ADAPTER, base_model_source, sft_adapter_path
from domain_lm import DomainLM, wrap

# "offline" = deterministic CPU simulator for local validation.
# "adapter" = real generation from the Assignment 1 QLoRA adapter (needs a GPU runtime).
BACKEND = "offline"
SFT_ADAPTER = DEFAULT_SFT_ADAPTER

ARTIFACTS = A2 / "outputs"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

lm = DomainLM(
    base_model_id=base_model_source(),
    adapter_path=sft_adapter_path(SFT_ADAPTER) if BACKEND == "adapter" else None,
    backend=BACKEND,
    label=f"sft_{SFT_ADAPTER}",
)
print(json.dumps(lm.describe(), indent=2))


## A1 — Five prompting strategies on five domain queries

| Strategy | Configuration | Expected effect |
|---|---|---|
| Zero-shot | No examples; task described in the instruction only | Tests raw adapter capability |
| Few-shot (3-shot) | 3 domain Q&A examples prepended | Improves format adherence |
| Instruction prompting | Explicit role + step-by-step instruction format | Improves structure |
| Self-consistency | Same prompt 5× at `temperature=0.7`, majority vote | Reduces variance |
| APE | 5 LLM-generated prompt variants, best selected by output quality | Optimises instruction wording |

Answers are scored by one shared rubric (`src/answer_scoring.py`) out of 100:
grounding 35, structure 20, completeness 20, specificity 15, non-repetition 10. The same scorer is
reused for ToT path selection in Part B and for the chosen/rejected decision in Part C, so the
preference labels are consistent with these tables.


In [ ]:
import prompt_strategies as ps
from assignment2_queries import DOMAIN_QUERIES

for i, q in enumerate(DOMAIN_QUERIES, 1):
    print(f"Q{i}. {q}")


In [ ]:
rows = ps.run_all_strategies(lm, DOMAIN_QUERIES)
print(f"{len(rows)} strategy runs = {len(DOMAIN_QUERIES)} queries x {len(ps.STRATEGIES)} strategies")


In [ ]:
import pandas as pd

table = pd.DataFrame([{
    "query": f"Q{DOMAIN_QUERIES.index(r['query']) + 1}",
    "strategy": r["strategy"],
    "score": r["score"]["total"],
    "grounding": r["score"]["grounding"],
    "structure": r["score"]["structure"],
    "words": r["score"]["words"],
    "answer": " ".join(r["answer"].split())[:110] + "...",
} for r in rows])
pivot = table.pivot(index="query", columns="strategy", values="score")
display(pivot)
display(pd.DataFrame(ps.strategy_summary(rows)))


In [ ]:
# Full text for one query so the qualitative difference is visible, not just the score.
focus = DOMAIN_QUERIES[1]
for r in [r for r in rows if r["query"] == focus]:
    print("=" * 100)
    print(r["strategy"], "| score", r["score"]["total"])
    print(wrap(r["answer"]))


In [ ]:
# Self-consistency detail: the 5 samples, the vote, and the agreement rate.
sc = ps.run_self_consistency(lm, focus)
print("agreement:", sc["agreement"], "| majority cluster size:", sc["majority_size"], "/ 5")
print("vote counts:", {k[:28] + "...": v for k, v in sc["vote_counts"].items()})
for i, sample in enumerate(sc["samples"], 1):
    print(f"\n--- sample {i} ---\n", wrap(sample[:400]))


In [ ]:
# APE detail: the generated prompt variants and the score of each resulting answer.
ape = ps.run_ape(lm, focus)
print("variants tried:", ape["variants_tried"], "| scores:", ape["variant_scores"])
print("\nRaw LLM proposals:\n", wrap(ape["raw_proposals"]))
print("\nSelected prompt:\n", wrap(ape["selected_prompt"]))


### A1 quality comparison — what to write up

Read the pivot table and the summary table, then record:

1. **Zero-shot** is the floor: it tests raw adapter capability and typically returns a short,
   unstructured definition with no limiting condition and no caveat.
2. **Few-shot (3-shot)** mainly buys *format adherence* — the answer starts to look like the
   exemplars — but it adds little new corpus content, so grounding moves less than structure.
3. **Instruction prompting** gives the largest single jump in the structure component because the
   numbered format is specified explicitly.
4. **Self-consistency** does not change the shape of a good answer; it removes the unlucky samples.
   Report the agreement rate: a high rate means the adapter is already stable on that query, so the
   extra 5× cost is not justified.
5. **APE** wins when the hand-written instruction was not optimal; compare `variant_scores` spread.
   A narrow spread means the task is insensitive to wording.

State the cost side too: self-consistency is 5 generations per answer and APE is 1 + N generations,
so both are far more expensive per query than instruction prompting.


## A2 — Prompt chaining and function calling

**Chain:** Step 1 extracts the relevant facts from the retrieved corpus context →
Step 2 synthesises the answer *only* from the Step 1 facts. The single-step prompt is run on the
same complex query so the improvement can be attributed to decomposition rather than to context.


In [ ]:
from assignment2_queries import CHAIN_QUERY, CLINICAL_FACT_SCHEMA

chain = ps.run_prompt_chain(lm, CHAIN_QUERY)
print("COMPLEX QUERY:\n", wrap(CHAIN_QUERY))
print("\n--- STEP 1: extracted facts ---\n", wrap(chain["step1_facts"]))
print("\n--- STEP 2: synthesised answer ---\n", wrap(chain["chained_answer"]))
print("\n--- SINGLE-STEP BASELINE ---\n", wrap(chain["single_step_answer"]))


In [ ]:
comparison = pd.DataFrame([
    {"approach": "single-step prompt", **chain["single_step_score"]},
    {"approach": "2-step chain", **chain["chained_score"]},
])
display(comparison)
delta = chain["chained_score"]["total"] - chain["single_step_score"]["total"]
print(f"Chaining changes the rubric score by {delta:+.2f} points.")


### Function calling / structured output

A JSON schema defines the domain output contract (`entity`, `value`, `unit`, `source_clause`,
`confidence`). The model is asked to return JSON only; the output is parsed and validated against
the schema, and a validation failure is fed back to the model as a repair turn. Required keys,
unexpected keys, and declared types are all checked — a raw `json.loads` would not catch those.


In [ ]:
print(json.dumps(CLINICAL_FACT_SCHEMA, indent=2))


In [ ]:
call = ps.run_function_call(lm, "Extract one clinical fact about diagnostic test performance.")
print("valid:", call["valid"], "| attempts:", len(call["attempts"]))
for attempt in call["attempts"]:
    print(f"\nattempt {attempt['attempt']} valid={attempt['valid']}"
          + (f" error={attempt.get('error')}" if not attempt["valid"] else ""))
    print(wrap(attempt["raw"][:400]))
print("\nParsed object:\n", json.dumps(call["parsed"], indent=2))


In [ ]:
# Negative control: the validator must reject malformed or off-schema output.
for bad in ['{"entity": "x"}', 'not json at all',
            '{"entity":"x","value":"y","unit":"z","source_clause":"c","confidence":"high"}']:
    try:
        ps.parse_structured_output(bad)
        print("ACCEPTED (unexpected):", bad)
    except ps.SchemaValidationError as exc:
        print("rejected:", exc)


In [ ]:
artefact = {
    "backend": lm.describe(),
    "a1_rows": [{k: v for k, v in r.items() if k != "samples"} for r in rows],
    "a1_summary": ps.strategy_summary(rows),
    "a2_chain": chain,
    "a2_function_call": call,
}
path = ARTIFACTS / "part_a_results.json"
path.write_text(json.dumps(artefact, indent=2, ensure_ascii=False), encoding="utf-8")
print("saved", path)


## Part A conclusion

- **Raw capability vs. scaffolding.** Zero-shot measures what the SFT adapter learned in
  Assignment 1; every other strategy measures how much of that knowledge the prompt can surface.
- **Structure is the cheapest win.** Instruction prompting costs one generation and recovers most
  of the gap, which is why it is the backbone of the APE variants and the Part B techniques.
- **Chaining beats a single step on decomposable queries** because Step 2 can only use facts that
  Step 1 committed to, which suppresses unsupported claims and keeps the answer on-corpus.
- **Function calling makes the output machine-checkable.** Schema validation converts a vague
  "looks right" judgement into a pass/fail gate, and the repair turn makes the gate recoverable.
- These outputs feed Part C directly: the zero-shot answers become `rejected` and the
  instruction / self-consistency / APE answers become `chosen`.
